# AIdanChem — ADMET 22 속성 예측 시연

이 노트북은 **AIdanChem** 모델을 사용하여 분자의 SMILES 표기를 입력받아 22가지 ADMET(흡수·분포·대사·배설·독성) 속성을 예측합니다.

## 예측 속성 (22가지)

| 범주 | 속성 | 유형 |
|------|------|------|
| **Absorption (흡수)** | Caco2, HIA, Pgp, Bioavailability, Lipophilicity, Solubility | 회귀/분류 혼합 |
| **Distribution (분포)** | BBB, PPBR, VDss | 분류/회귀 혼합 |
| **Metabolism (대사)** | CYP2C9, CYP2D6, CYP3A4, CYP2C9_Substrate, CYP2D6_Substrate, CYP3A4_Substrate | 분류 |
| **Excretion (배설)** | Half_Life, Clearance_Hepatocyte, Clearance_Microsome | 회귀 |
| **Toxicity (독성)** | LD50, hERG, AMES, DILI | 회귀/분류 혼합 |

## 필요 파일
- `outputs/checkpoint_best.pth` — 학습된 모델 체크포인트 (model_state_dict, mean, std 포함)
- *(선택)* `outputs/label_mean_std.npz` — 정규화 통계값 (체크포인트에 없는 경우)


## 1. 패키지 설치

In [ ]:
# 필요한 패키지가 없는 경우 아래 주석을 해제하고 실행하세요
# !pip install torch transformers rdkit-pypi pandas numpy matplotlib seaborn ipywidgets

## 2. 라이브러리 임포트

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModelForMaskedLM, AutoTokenizer

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec

# RDKit (SMILES 유효성 검사용, 선택적)
try:
    from rdkit import Chem
    from rdkit.Chem import Draw, rdMolDescriptors
    from rdkit.Chem.Draw import IPythonConsole
    from IPython.display import display
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("[INFO] RDKit 없음 — SMILES 유효성 검사 및 구조 시각화를 건너뜁니다.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

## 3. 설정 (경로 및 모델 선택)

> **여기에서 체크포인트 경로와 베이스 모델을 지정하세요.**

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────
CHECKPOINT_PATH = "outputs/checkpoint_best.pth"   # 학습된 모델 체크포인트
MEAN_STD_PATH   = "outputs/label_mean_std.npz"    # (선택) 정규화 통계 파일

# ── 베이스 모델 선택 ───────────────────────────────────────
# 학습 시 사용한 베이스 모델과 동일해야 합니다
BASE_MODEL_OPTIONS = {
    "deberta" : "sagawa/ZINC-deberta",
    "roberta" : "entropy/roberta_zinc_480m",
    "bert"    : "unikei/bert-base-smiles",
    "chemberta": "DeepChem/ChemBERTa-77M-MLM",
}

MODEL_KEY       = "deberta"                        # deberta / roberta / bert / chemberta
BASE_MODEL_NAME = BASE_MODEL_OPTIONS[MODEL_KEY]

NUM_CLASSES  = 22
MAX_LENGTH   = 512
BATCH_SIZE   = 32

print(f"베이스 모델  : {BASE_MODEL_NAME}")
print(f"체크포인트   : {CHECKPOINT_PATH}")

## 4. ADMET 속성 메타데이터 정의

In [ ]:
FEATURE_NAMES = [
    "Caco2", "HIA", "Pgp", "Bioavailability", "Lipophilicity", "Solubility",
    "BBB", "PPBR", "VDss",
    "CYP2C9", "CYP2D6", "CYP3A4", "CYP2C9_Substrate", "CYP2D6_Substrate", "CYP3A4_Substrate",
    "Half_Life", "Clearance_Hepatocyte", "Clearance_Microsome",
    "LD50", "hERG", "AMES", "DILI",
]

# 각 속성의 메타데이터: (카테고리, 유형, 단위/설명, 높은 값의 의미)
ADMET_META = {
    # ── Absorption ─────────────────────────────────────────
    "Caco2"           : {"cat": "Absorption",    "type": "regression",     "unit": "log cm/s",        "high_means": "높은 장 투과성"},
    "HIA"             : {"cat": "Absorption",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "장 흡수 양성"},
    "Pgp"             : {"cat": "Absorption",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "P-gp 기질 양성"},
    "Bioavailability" : {"cat": "Absorption",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "경구 생체이용률 양성"},
    "Lipophilicity"   : {"cat": "Absorption",    "type": "regression",     "unit": "log D (pH 7.4)",  "high_means": "지용성 높음"},
    "Solubility"      : {"cat": "Absorption",    "type": "regression",     "unit": "log mol/L",       "high_means": "수용성 높음"},
    # ── Distribution ───────────────────────────────────────
    "BBB"             : {"cat": "Distribution",  "type": "classification", "unit": "확률 (0–1)",       "high_means": "BBB 투과 양성"},
    "PPBR"            : {"cat": "Distribution",  "type": "regression",     "unit": "%",               "high_means": "혈장 단백 결합률 높음"},
    "VDss"            : {"cat": "Distribution",  "type": "regression",     "unit": "log L/kg",        "high_means": "분포용적 높음"},
    # ── Metabolism ─────────────────────────────────────────
    "CYP2C9"          : {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP2C9 억제제 양성"},
    "CYP2D6"          : {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP2D6 억제제 양성"},
    "CYP3A4"          : {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP3A4 억제제 양성"},
    "CYP2C9_Substrate": {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP2C9 기질 양성"},
    "CYP2D6_Substrate": {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP2D6 기질 양성"},
    "CYP3A4_Substrate": {"cat": "Metabolism",    "type": "classification", "unit": "확률 (0–1)",       "high_means": "CYP3A4 기질 양성"},
    # ── Excretion ──────────────────────────────────────────
    "Half_Life"            : {"cat": "Excretion", "type": "regression", "unit": "log hr",                    "high_means": "반감기 길음"},
    "Clearance_Hepatocyte" : {"cat": "Excretion", "type": "regression", "unit": "log µL/min/10⁶ cells",      "high_means": "간세포 청소율 높음"},
    "Clearance_Microsome"  : {"cat": "Excretion", "type": "regression", "unit": "log mL/min/g",              "high_means": "마이크로솜 청소율 높음"},
    # ── Toxicity ───────────────────────────────────────────
    "LD50" : {"cat": "Toxicity", "type": "regression",     "unit": "log mg/kg",   "high_means": "급성 독성 낮음 (더 안전)"},
    "hERG" : {"cat": "Toxicity", "type": "classification", "unit": "확률 (0–1)",  "high_means": "hERG 억제 양성 (심독성 위험)"},
    "AMES" : {"cat": "Toxicity", "type": "classification", "unit": "확률 (0–1)",  "high_means": "변이원성 양성"},
    "DILI" : {"cat": "Toxicity", "type": "classification", "unit": "확률 (0–1)",  "high_means": "약물유발 간독성 양성"},
}

CATEGORY_COLORS = {
    "Absorption"  : "#4C72B0",
    "Distribution": "#55A868",
    "Metabolism"  : "#C44E52",
    "Excretion"   : "#8172B2",
    "Toxicity"    : "#CCB974",
}

print(f"총 예측 속성 수: {len(FEATURE_NAMES)}")
for cat, color in CATEGORY_COLORS.items():
    props = [k for k, v in ADMET_META.items() if v['cat'] == cat]
    print(f"  {cat:15s}: {', '.join(props)}")

## 5. 모델 클래스 정의

In [ ]:
class CustomRegModel(nn.Module):
    """AIdanChem 멀티태스크 ADMET 예측 모델.

    SMILES 토큰 시퀀스를 화학 언어 모델(DeBERTa/RoBERTa/BERT)로 인코딩한 후
    Mean Pooling → MLP 헤드를 통해 22가지 ADMET 속성을 동시에 예측합니다.
    """

    def __init__(self, base_model_name: str, num_classes: int = 22):
        super().__init__()
        self.base = AutoModelForMaskedLM.from_pretrained(base_model_name)
        self.model_type = self.base.config.model_type

        hidden_size = self.base.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        if self.model_type == "bert":
            outputs = self.base.bert(input_ids=input_ids, attention_mask=attention_mask)
        elif self.model_type == "roberta":
            outputs = self.base.roberta(input_ids=input_ids, attention_mask=attention_mask)
        elif self.model_type == "deberta":
            outputs = self.base.deberta(input_ids=input_ids, attention_mask=attention_mask)
        else:
            raise ValueError(f"지원하지 않는 모델 유형: {self.model_type}")

        last_hidden = outputs.last_hidden_state                        # (B, L, H)
        mask_expanded = attention_mask.unsqueeze(-1)                   # (B, L, 1)
        sum_hidden = (last_hidden * mask_expanded).sum(dim=1)          # (B, H)
        valid_tokens = attention_mask.sum(dim=1, keepdim=True).float() # (B, 1)
        mean_hidden = sum_hidden / valid_tokens                        # (B, H)

        return self.regressor(mean_hidden)                             # (B, 22)


print("CustomRegModel 클래스 정의 완료")

## 6. 모델 로드

In [ ]:
def load_model_from_checkpoint(
    checkpoint_path: str,
    base_model_name: str,
    num_classes: int = 22,
    mean_std_path: str | None = None,
    device: torch.device = torch.device("cpu"),
):
    """체크포인트에서 모델, 토크나이저, 정규화 통계를 불러옵니다.

    Returns
    -------
    model        : CustomRegModel (eval 모드)
    tokenizer    : AutoTokenizer
    mean_values  : np.ndarray (22,)
    std_values   : np.ndarray (22,)
    """
    print(f"[1/3] 베이스 모델 로드: {base_model_name}")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = CustomRegModel(base_model_name, num_classes=num_classes)

    print(f"[2/3] 체크포인트 로드: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # ── state_dict 분리 적용 ──────────────────────────────
    state_dict = checkpoint["model_state_dict"]
    base_sd, reg_sd = {}, {}
    for key, val in state_dict.items():
        if key.startswith("base."):
            base_sd[key[len("base."):]] = val
        elif key.startswith("regressor."):
            reg_sd[key[len("regressor."):]] = val
        else:
            base_sd[key] = val

    model.base.load_state_dict(base_sd)
    model.regressor.load_state_dict(reg_sd)
    model.to(device).eval()

    # ── 정규화 통계 불러오기 ──────────────────────────────
    print("[3/3] 정규화 통계 로드")
    if "mean" in checkpoint and "std" in checkpoint:
        mean_values = np.array(checkpoint["mean"], dtype=np.float32)
        std_values  = np.array(checkpoint["std"],  dtype=np.float32)
    elif mean_std_path and os.path.exists(mean_std_path):
        data = np.load(mean_std_path)
        mean_values = data["mean"].astype(np.float32)
        std_values  = data["std"].astype(np.float32)
    else:
        raise FileNotFoundError(
            "정규화 통계를 찾을 수 없습니다.\n"
            "체크포인트에 'mean'/'std' 키가 있거나 mean_std_path를 지정하세요."
        )

    print(f"모델 로드 완료 (파라미터: {sum(p.numel() for p in model.parameters()):,}개)")
    return model, tokenizer, mean_values, std_values


# ── 실제 로드 실행 ─────────────────────────────────────────────────
model, tokenizer, mean_values, std_values = load_model_from_checkpoint(
    checkpoint_path = CHECKPOINT_PATH,
    base_model_name = BASE_MODEL_NAME,
    num_classes     = NUM_CLASSES,
    mean_std_path   = MEAN_STD_PATH,
    device          = device,
)

## 7. 예측 함수 정의

In [ ]:
def validate_smiles(smiles_list: list[str]) -> list[str | None]:
    """RDKit으로 SMILES 유효성 확인. RDKit 미설치 시 통과."""
    if not RDKIT_AVAILABLE:
        return smiles_list
    validated = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f"  [경고] 유효하지 않은 SMILES: {smi!r}")
            validated.append(None)
        else:
            validated.append(Chem.MolToSmiles(mol))  # canonical SMILES
    return validated


@torch.no_grad()
def predict_admet(
    model: CustomRegModel,
    tokenizer,
    smiles_list: list[str],
    mean_values: np.ndarray,
    std_values: np.ndarray,
    feature_names: list[str],
    admet_meta: dict,
    max_length: int = 512,
    batch_size: int = 32,
    device: torch.device = torch.device("cpu"),
) -> pd.DataFrame:
    """SMILES 목록에 대해 22가지 ADMET 속성을 예측합니다.

    Parameters
    ----------
    smiles_list : 예측할 SMILES 문자열 목록

    Returns
    -------
    pd.DataFrame : SMILES + 22개 속성 예측값 (역정규화 완료)
    """
    model.eval()
    all_preds = []

    for i in range(0, len(smiles_list), batch_size):
        batch_smiles = smiles_list[i : i + batch_size]

        encoded = tokenizer(
            batch_smiles,
            max_length     = max_length,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        input_ids      = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        raw_preds = model(input_ids, attention_mask).cpu().numpy()  # (B, 22)
        all_preds.append(raw_preds)

    raw_preds_all = np.concatenate(all_preds, axis=0)  # (N, 22)

    # ── 역정규화 & 분류 속성 sigmoid ──────────────────────
    denorm_preds = np.zeros_like(raw_preds_all)
    for j, feat in enumerate(feature_names):
        meta = admet_meta[feat]
        if meta["type"] == "classification":
            # 이진 분류: 정규화된 로짓 → sigmoid로 확률 변환
            denorm_preds[:, j] = 1.0 / (1.0 + np.exp(-raw_preds_all[:, j]))
        else:
            # 회귀: Z-score 역정규화
            denorm_preds[:, j] = raw_preds_all[:, j] * std_values[j] + mean_values[j]

    results_df = pd.DataFrame(denorm_preds, columns=feature_names)
    results_df.insert(0, "SMILES", smiles_list)
    return results_df


print("predict_admet 함수 정의 완료")

## 8. 결과 출력 및 시각화 함수

In [ ]:
def format_result_table(df_row: pd.Series, admet_meta: dict, feature_names: list[str]) -> pd.DataFrame:
    """단일 분자 예측 결과를 가독성 있는 DataFrame으로 변환합니다."""
    rows = []
    for feat in feature_names:
        meta  = admet_meta[feat]
        value = df_row[feat]
        if meta["type"] == "classification":
            label = "양성 (+)" if value >= 0.5 else "음성 (−)"
            disp  = f"{value:.3f}  →  {label}"
        else:
            disp = f"{value:.4f}"
        rows.append({
            "속성"      : feat,
            "카테고리"  : meta["cat"],
            "유형"      : meta["type"],
            "단위"      : meta["unit"],
            "예측값"    : disp,
            "해석"      : meta["high_means"],
        })
    return pd.DataFrame(rows)


def plot_admet_result(
    df_row: pd.Series,
    admet_meta: dict,
    feature_names: list[str],
    category_colors: dict,
    molecule_name: str = "분자",
    figsize: tuple = (18, 10),
):
    """ADMET 예측 결과를 카테고리별 막대 그래프로 시각화합니다."""
    categories = ["Absorption", "Distribution", "Metabolism", "Excretion", "Toxicity"]

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"ADMET 22 속성 예측  —  {molecule_name}", fontsize=15, fontweight="bold", y=1.01)

    n_cats = len(categories)
    gs = gridspec.GridSpec(1, n_cats, figure=fig, wspace=0.35)

    for ax_idx, cat in enumerate(categories):
        ax    = fig.add_subplot(gs[ax_idx])
        feats = [f for f in feature_names if admet_meta[f]["cat"] == cat]
        vals  = [df_row[f] for f in feats]
        types = [admet_meta[f]["type"] for f in feats]
        color = category_colors[cat]

        bar_colors = []
        for v, t in zip(vals, types):
            if t == "classification":
                bar_colors.append("#E74C3C" if v >= 0.5 else "#2ECC71")
            else:
                bar_colors.append(color)

        bars = ax.barh(range(len(feats)), vals, color=bar_colors, edgecolor="white", linewidth=0.5)

        # 값 레이블
        for i, (bar, v, t) in enumerate(zip(bars, vals, types)):
            label = f"{v:.3f}" if t == "classification" else f"{v:.3f}"
            ax.text(
                bar.get_width() + abs(ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.02,
                bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left", fontsize=8,
            )

        ax.set_yticks(range(len(feats)))
        ax.set_yticklabels([f.replace("_", "\n", 1) for f in feats], fontsize=8)
        ax.set_title(cat, fontsize=10, fontweight="bold", color=color, pad=6)
        ax.set_xlabel("예측값", fontsize=8)
        ax.invert_yaxis()
        ax.spines[["top", "right"]].set_visible(False)

        # 분류 속성에 0.5 기준선
        if any(t == "classification" for t in types):
            ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

    # 범례
    legend_patches = [
        mpatches.Patch(color="#2ECC71", label="분류: 음성 (< 0.5)"),
        mpatches.Patch(color="#E74C3C", label="분류: 양성 (≥ 0.5)"),
    ] + [
        mpatches.Patch(color=c, label=f"회귀: {cat}") for cat, c in category_colors.items()
        if any(admet_meta[f]["type"] == "regression" and admet_meta[f]["cat"] == cat for f in feature_names)
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.07), fontsize=8, framealpha=0.8)

    plt.tight_layout()
    plt.show()


print("시각화 함수 정의 완료")

## 9. 단일 분자 예측 — 아스피린 (Aspirin)

아스피린(아세틸살리실산)의 SMILES로 테스트합니다.

In [ ]:
# ── 예측할 분자 설정 ──────────────────────────────────────────────
demo_smiles  = "CC(=O)Oc1ccccc1C(=O)O"  # 아스피린
demo_name    = "Aspirin (아스피린)"

# SMILES 유효성 검사
validated = validate_smiles([demo_smiles])
if validated[0] is None:
    raise ValueError("유효하지 않은 SMILES입니다.")

# 분자 구조 출력 (RDKit 있는 경우)
if RDKIT_AVAILABLE:
    mol = Chem.MolFromSmiles(demo_smiles)
    print(f"분자식: {rdMolDescriptors.CalcMolFormula(mol)}")
    print(f"분자량: {rdMolDescriptors.CalcExactMolWt(mol):.3f} Da")
    img = Draw.MolToImage(mol, size=(300, 200))
    display(img)
else:
    print(f"SMILES: {demo_smiles}")

print(f"\n{demo_name} ADMET 예측 중...")

In [ ]:
# ── ADMET 예측 실행 ─────────────────────────────────────────────
results_df = predict_admet(
    model         = model,
    tokenizer     = tokenizer,
    smiles_list   = [demo_smiles],
    mean_values   = mean_values,
    std_values    = std_values,
    feature_names = FEATURE_NAMES,
    admet_meta    = ADMET_META,
    max_length    = MAX_LENGTH,
    batch_size    = BATCH_SIZE,
    device        = device,
)

# ── 결과 테이블 출력 ────────────────────────────────────────────
table = format_result_table(results_df.iloc[0], ADMET_META, FEATURE_NAMES)

print(f"\n{'='*70}")
print(f"  {demo_name} — ADMET 22 속성 예측 결과")
print(f"{'='*70}")
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 120)
display(table.style
    .set_properties(**{"text-align": "left"})
    .apply(lambda s: [
        f"background-color: {CATEGORY_COLORS.get(v, 'white')}22"
        for v in s
    ], subset=["카테고리"])
    .set_caption(f"{demo_name} ADMET 예측 결과")
    .hide(axis="index")
)

In [ ]:
# ── 시각화 ─────────────────────────────────────────────────────
plot_admet_result(
    df_row          = results_df.iloc[0],
    admet_meta      = ADMET_META,
    feature_names   = FEATURE_NAMES,
    category_colors = CATEGORY_COLORS,
    molecule_name   = demo_name,
)

## 10. 나만의 SMILES 입력

아래 셀에서 `my_smiles`를 원하는 분자의 SMILES로 교체하세요.

In [ ]:
# ── 원하는 SMILES로 교체하세요 ─────────────────────────────────
my_smiles = "CC(=O)Oc1ccccc1C(=O)O"   # 여기를 수정하세요
my_name   = "My Molecule"              # 분자 이름 (선택)

validated = validate_smiles([my_smiles])
if validated[0] is None:
    raise ValueError("유효하지 않은 SMILES입니다.")

if RDKIT_AVAILABLE:
    mol = Chem.MolFromSmiles(my_smiles)
    display(Draw.MolToImage(mol, size=(300, 200)))

results = predict_admet(
    model=model, tokenizer=tokenizer,
    smiles_list=[my_smiles],
    mean_values=mean_values, std_values=std_values,
    feature_names=FEATURE_NAMES, admet_meta=ADMET_META,
    device=device,
)

display(format_result_table(results.iloc[0], ADMET_META, FEATURE_NAMES)
    .style.hide(axis="index").set_caption(my_name))

plot_admet_result(results.iloc[0], ADMET_META, FEATURE_NAMES,
                 CATEGORY_COLORS, molecule_name=my_name)

## 11. 다중 분자 배치 예측

여러 분자를 한 번에 예측하고 히트맵으로 비교합니다.

In [ ]:
# ── 배치 예측 예시 분자 목록 ──────────────────────────────────
DEMO_MOLECULES = [
    {"name": "Aspirin",      "smiles": "CC(=O)Oc1ccccc1C(=O)O"},
    {"name": "Ibuprofen",    "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O"},
    {"name": "Caffeine",     "smiles": "Cn1cnc2c1c(=O)n(c(=O)n2C)C"},
    {"name": "Paracetamol",  "smiles": "CC(=O)Nc1ccc(O)cc1"},
    {"name": "Ciprofloxacin","smiles": "O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O"},
    {"name": "Metformin",    "smiles": "CN(C)C(=N)NC(=N)N"},
    {"name": "Atorvastatin", "smiles": "CC(C)c1c(C(=O)Nc2ccccc2)c(c(c(n1)CC[C@@H](O)C[C@@H](O)CC(=O)O)c3ccc(F)cc3)c4ccccc4"},
    {"name": "Sildenafil",   "smiles": "CCCC1=NN(C2=CC(=C(C=C2)S(=O)(=O)N2CCN(CC2)C)OCC)C(=O)C1"},
]

batch_smiles = [m["smiles"] for m in DEMO_MOLECULES]
batch_names  = [m["name"]   for m in DEMO_MOLECULES]

print(f"{len(batch_smiles)}개 분자 ADMET 예측 중...")
batch_results = predict_admet(
    model=model, tokenizer=tokenizer,
    smiles_list=batch_smiles,
    mean_values=mean_values, std_values=std_values,
    feature_names=FEATURE_NAMES, admet_meta=ADMET_META,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE, device=device,
)
batch_results.insert(1, "Name", batch_names)
print("예측 완료!")
display(batch_results.set_index("Name")[FEATURE_NAMES].round(3))

In [ ]:
# ── 히트맵으로 다중 분자 비교 ────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

heatmap_data = batch_results.set_index("Name")[FEATURE_NAMES].astype(float)

# 분류 속성과 회귀 속성을 분리해 각각 정규화
cls_feats = [f for f in FEATURE_NAMES if ADMET_META[f]["type"] == "classification"]
reg_feats = [f for f in FEATURE_NAMES if ADMET_META[f]["type"] == "regression"]

normed = heatmap_data.copy()
for feat in reg_feats:
    col = heatmap_data[feat]
    rng = col.max() - col.min()
    normed[feat] = (col - col.min()) / rng if rng > 1e-8 else 0.5

fig, ax = plt.subplots(figsize=(18, 5))

category_order = FEATURE_NAMES
cat_colors_list = [CATEGORY_COLORS[ADMET_META[f]["cat"]] for f in category_order]

sns.heatmap(
    normed[category_order],
    ax       = ax,
    cmap     = "RdYlGn_r",
    vmin     = 0, vmax = 1,
    annot    = heatmap_data[category_order].round(2),
    fmt      = ".2f",
    linewidths = 0.3,
    linecolor  = "white",
    annot_kws  = {"size": 7},
    cbar_kws   = {"label": "정규화된 예측값 (0=낮음, 1=높음)"},
)

# 카테고리 색상 축 레이블
for tick, feat in zip(ax.get_xticklabels(), category_order):
    tick.set_color(CATEGORY_COLORS[ADMET_META[feat]["cat"]])
    tick.set_fontsize(8)

ax.set_title("ADMET 22 속성 — 다중 분자 비교 히트맵", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("분자", fontsize=10)
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")

# 카테고리 구분선
cat_boundaries = []
prev_cat = None
for i, feat in enumerate(category_order):
    cur_cat = ADMET_META[feat]["cat"]
    if cur_cat != prev_cat:
        cat_boundaries.append(i)
    prev_cat = cur_cat
for boundary in cat_boundaries[1:]:
    ax.axvline(boundary, color="black", linewidth=2)

# 카테고리 레이블
cat_ranges = {}
for i, feat in enumerate(category_order):
    cat = ADMET_META[feat]["cat"]
    if cat not in cat_ranges:
        cat_ranges[cat] = [i, i]
    cat_ranges[cat][1] = i

for cat, (start, end) in cat_ranges.items():
    mid = (start + end) / 2 + 0.5
    ax.text(mid, -0.6, cat, ha="center", va="top",
            fontsize=9, fontweight="bold",
            color=CATEGORY_COLORS[cat],
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig("admet_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("히트맵이 'admet_heatmap.png'에 저장되었습니다.")

In [ ]:
# ── CSV로 결과 저장 ─────────────────────────────────────────
output_csv = "admet_predictions.csv"
batch_results.to_csv(output_csv, index=False, float_format="%.4f")
print(f"예측 결과가 '{output_csv}'에 저장되었습니다.")
display(batch_results.set_index("Name")[FEATURE_NAMES].round(4))

---

## 참고: 속성별 해석 가이드

### 분류 속성 (확률 0–1, 임계값 0.5)

| 속성 | 양성(+) 의미 | 약물 개발 관점 |
|------|------------|---------------|
| HIA | 장 흡수 잘 됨 | 경구 투여에 유리 |
| Pgp | P-gp 기질 | 장내 배출 증가, 흡수 감소 가능 |
| Bioavailability | 경구 생체이용률 높음 | 경구 투여에 적합 |
| BBB | 혈뇌장벽 통과 | CNS 표적에 유리, 비CNS엔 위험 |
| CYP2C9/2D6/3A4 | CYP 억제 | 약물 상호작용 위험 |
| CYP*_Substrate | CYP 기질 | 해당 효소에 의해 대사됨 |
| hERG | hERG 억제 | 심독성(QT 연장) 위험 |
| AMES | 변이원성 | 발암 가능성 |
| DILI | 약물유발 간독성 | 간 독성 위험 |

### 회귀 속성

| 속성 | 단위 | 이상적인 범위 |
|------|------|---------------|
| Caco2 | log cm/s | > −5.15 (양호) |
| Lipophilicity | log D (pH 7.4) | 0–3 (최적) |
| Solubility | log mol/L | > −4 (양호) |
| PPBR | % | < 90% (낮을수록 유리) |
| VDss | log L/kg | 0.04–20 L/kg |
| Half_Life | log hr | 화합물에 따라 다름 |
| LD50 | log mg/kg | 높을수록 안전 |

---
*AIdanChem: Fine-tuned language models for ADMET prediction*